In [8]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer

# Import and select the data

In [9]:
df = pd.read_csv("data/combined_h1_2018_2025_excl_nan.csv")
df.head()

,Datetime,Location,NO2,O3,PM10,Hr,RainDur,T,WVv,StrGlo,Year,Date,Time,Month,Month_str,Weekday,Weekday_str,Day,YearMonth,meanVehicles
0,2017-12-31 23:00:00+00:00,Zch_Rosengartenstrasse,17.20,44.33,18.81,67.36,0.00,9.02,1.85,0.02,2018.0,2017-12-31,23:00:00,12.0,Dec,6.0,Sun,31.0,2018-12-01,538.5
1,2017-12-31 23:00:00+00:00,Zch_Schimmelstrasse,9.33,49.89,16.57,64.18,0.00,9.60,0.70,0.02,2018.0,2017-12-31,23:00:00,12.0,Dec,6.0,Sun,31.0,2018-12-01,340.0
2,2017-12-31 23:00:00+00:00,Zch_Stampfenbachstrasse,12.80,46.59,27.35,66.37,0.00,9.06,4.62,0.02,2018.0,2017-12-31,23:00:00,12.0,Dec,6.0,Sun,31.0,2018-12-01,197.5
3,2018-01-01 00:00:00+00:00,Zch_Rosengartenstrasse,24.10,41.80,19.33,69.54,0.27,9.28,1.59,0.01,2018.0,2018-01-01,00:00:00,1.0,Jan,0.0,Mon,1.0,2018-01-01,1029.0
4,2018-01-01 00:00:00+00:00,Zch_Schimmelstrasse,8.67,50.66,11.73,67.29,0.00,9.53,0.99,0.01,2018.0,2018-01-01,00:00:00,1.0,Jan,0.0,Mon,1.0,2018-01-01,621.5


In [10]:
# Re-arrange columns
cols = ["Datetime", "YearMonth", "Date", "Year", "Month", "Month_str", "Weekday", "Weekday_str", "Day", "Time",
        "Location",
        "NO2", "O3", "PM10",
        "Hr", "RainDur", "T", "WVv", "StrGlo",
        "meanVehicles"]

df = df[cols]
df.head()

,Datetime,YearMonth,Date,Year,Month,Month_str,Weekday,Weekday_str,Day,Time,Location,NO2,O3,PM10,Hr,RainDur,T,WVv,StrGlo,meanVehicles
0,2017-12-31 23:00:00+00:00,2018-12-01,2017-12-31,2018.0,12.0,Dec,6.0,Sun,31.0,23:00:00,Zch_Rosengartenstrasse,17.20,44.33,18.81,67.36,0.00,9.02,1.85,0.02,538.5
1,2017-12-31 23:00:00+00:00,2018-12-01,2017-12-31,2018.0,12.0,Dec,6.0,Sun,31.0,23:00:00,Zch_Schimmelstrasse,9.33,49.89,16.57,64.18,0.00,9.60,0.70,0.02,340.0
2,2017-12-31 23:00:00+00:00,2018-12-01,2017-12-31,2018.0,12.0,Dec,6.0,Sun,31.0,23:00:00,Zch_Stampfenbachstrasse,12.80,46.59,27.35,66.37,0.00,9.06,4.62,0.02,197.5
3,2018-01-01 00:00:00+00:00,2018-01-01,2018-01-01,2018.0,1.0,Jan,0.0,Mon,1.0,00:00:00,Zch_Rosengartenstrasse,24.10,41.80,19.33,69.54,0.27,9.28,1.59,0.01,1029.0
4,2018-01-01 00:00:00+00:00,2018-01-01,2018-01-01,2018.0,1.0,Jan,0.0,Mon,1.0,00:00:00,Zch_Schimmelstrasse,8.67,50.66,11.73,67.29,0.00,9.53,0.99,0.01,621.5


In [11]:
# Select columns
cols_selected = ["Year", "Month", "Weekday", "Time", "Day",
                 "Location",
                 "NO2", "O3", "PM10",
                 "Hr", "RainDur", "T", "WVv", "StrGlo",
                 "meanVehicles"]

df_selected = df[cols_selected]

# Make sure the time features are numbers, not strings
df_ready = df_selected.copy()
df_ready["Time"] = pd.to_datetime(df_ready["Time"], format="%H:%M:%S").dt.hour.astype("int64")
df_ready["Month"] = df_selected["Month"].astype(int)
df_ready["Day"] = df_selected["Day"].astype(int)
df_ready["Weekday"] = df_selected["Weekday"].astype(int)

df_ready.head()

,Year,Month,Weekday,Time,Day,Location,NO2,O3,PM10,Hr,RainDur,T,WVv,StrGlo,meanVehicles
0,2018.0,12,6,23,31,Zch_Rosengartenstrasse,17.20,44.33,18.81,67.36,0.00,9.02,1.85,0.02,538.5
1,2018.0,12,6,23,31,Zch_Schimmelstrasse,9.33,49.89,16.57,64.18,0.00,9.60,0.70,0.02,340.0
2,2018.0,12,6,23,31,Zch_Stampfenbachstrasse,12.80,46.59,27.35,66.37,0.00,9.06,4.62,0.02,197.5
3,2018.0,1,0,0,1,Zch_Rosengartenstrasse,24.10,41.80,19.33,69.54,0.27,9.28,1.59,0.01,1029.0
4,2018.0,1,0,0,1,Zch_Schimmelstrasse,8.67,50.66,11.73,67.29,0.00,9.53,0.99,0.01,621.5


# Prepare transformer - to prepare data for ML

In [12]:
# Preprocessing - OneHotEncoder, StandardScaler, Trigonometric encoding and SplineTransformer
# Prepare data transformer to be used in pipeline - do not use on the whole dataset!
# Encodings must be computed only on train data and then only applied on test data
# Trigonometric encoding could be in theory be done on the whole dataset, but for simplicity it is combined to have only one transformer

# Prepare Trigonometric encoding for time (safe before split)
def sin_transformer(period):
    return FunctionTransformer(lambda x: np.sin(x / period * 2 * np.pi))

def cos_transformer(period):
    return FunctionTransformer(lambda x: np.cos(x / period * 2 * np.pi))

# Prepare Spline Transformer
def periodic_spline_transformer(period, n_splines):
    degree = 3
    n_knots = n_splines + 1  # since include_bias is True
    return SplineTransformer(
        degree=degree,
        n_knots=n_knots,
        knots=np.linspace(0, period, n_knots).reshape(n_knots, 1),
        extrapolation="periodic",
        include_bias=True,
    )

# Prepare columns to be used
numeric_variables = ["Hr", "RainDur", "T", "WVv", "StrGlo", "meanVehicles"]

preprocess = ColumnTransformer(
    transformers=[
        ("Month_sin", sin_transformer(12), ["Month"]),
        ("Month_cos", cos_transformer(12), ["Month"]),
        ("Day_sin", sin_transformer(7), ["Day"]),
        ("Day_cos", cos_transformer(7), ["Day"]),
        ("Weekday_sin", sin_transformer(7), ["Weekday"]),
        ("Weekday_cos", cos_transformer(7), ["Weekday"]),
        ("Time_sin", sin_transformer(24), ["Time"]),
        ("Time_cos", cos_transformer(24), ["Time"]),
        ("Spline_month", periodic_spline_transformer(12, n_splines=5), ["Month"]),
        ("Spline_day", periodic_spline_transformer(7, n_splines=6), ["Day"]),
        ("Spline_weekday", periodic_spline_transformer(7, n_splines=6), ["Weekday"]),
        ("Spline_time", periodic_spline_transformer(24, n_splines=24), ["Time"]),
        ("encoded", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["Location"]),
        ("scaled", StandardScaler(), numeric_variables),
    ],
    remainder="passthrough"
).set_output(transform="pandas")

# Split for Train and Test - fit and transform the Preprocessing transformer

In [13]:
# Split manually based on year (to make sure the whole year is always included)
X_df = df_ready.drop(columns=["NO2", "O3", "PM10"]).copy()
y_df = df_ready[["NO2", "O3", "PM10"]].copy()

# Training datasets contain years 2018-2024
X_train = X_df[X_df["Year"].between(2018, 2024)].copy()
X_test = X_df[X_df["Year"] == 2025].copy()
# Testing datasets contain year 2025
y_train = y_df[X_df["Year"].between(2018, 2024)].copy()
y_test = y_df[X_df["Year"] == 2025].copy()

preprocess.fit(X_train)
X_train_transformed = preprocess.transform(X_train)
X_test_transformed = preprocess.transform(X_test)

In [14]:
X_train_transformed.to_csv("data/Datasets_MLready/X_train_transformed.csv", index=False)
X_test_transformed.to_csv("data/Datasets_MLready/X_test_transformed.csv", index=False)
y_train.to_csv("data/Datasets_MLready/y_train.csv", index=False)
y_test.to_csv("data/Datasets_MLready/y_test.csv", index=False)